# Patient Health Risk Segmentation Using Clustering
### Machine Learning Case Study – B.Tech CSE

---

## Project Overview
This project presents a complete, rigorous unsupervised machine learning pipeline designed to segment health insurance policyholders into meaningful health-indicator cohorts. Using physiological, clinical, and lifestyle measurements, the system uncovers natural groupings to assist healthcare organizations in tailoring preventive wellness and disease-mitigation initiatives.

# 2. Problem Statement

A health insurance provider wants to segment policyholders according to health-related indicators such as:
* **Age** (years)
* **BMI** (Body Mass Index in $\text{kg/m}^2$)
* **Blood Pressure** (Systolic and Diastolic in $\text{mmHg}$)
* **Total Cholesterol** ($\text{mg/dL}$)
* **Physical Activity Level** (exercise hours per week)
* **Previous Claims** (historical claim count)

The primary business objective is to identify groups with similar health-indicator patterns to support the proactive design of **preventive health programmes** (e.g., lifestyle guidance, cardiovascular conditioning, wellness screening).

> **Important Academic & Educational Disclaimer:**  
> This project uses synthetic data generated strictly for academic and educational purposes. The outputs of this model represent statistical clusters and must **NOT** be interpreted as medical diagnoses, clinical risk assessments, or criteria for insurance premium decisions or coverage denial.

# 3. Project Objectives

The project accomplishes the following objectives:
* Perform exploratory data analysis (EDA) to understand feature distributions and associations.
* Generate a realistic synthetic health dataset of 1,200 policyholders with embedded latent profiles.
* Handle missing clinical measurements using robust median imputation without data deletion.
* Apply feature scaling using `StandardScaler` to normalize physical measurement units.
* Apply K-Means clustering across candidate values of $K \in [2, 8]$.
* Apply Agglomerative Hierarchical Clustering using Ward linkage as a comparative benchmark.
* Determine the optimal number of clusters quantitatively and qualitatively.
* Use the Elbow Method and Within-Cluster Sum of Squares (WCSS).
* Use Silhouette Score analysis to evaluate cluster cohesion and separation.
* Generate a Dendrogram to visualize hierarchical merges and linkage distances.
* Apply Principal Component Analysis (PCA) for 2D dimensionality reduction and visualization.
* Visualize clusters in 2D PCA space with cluster centroids.
* Profile and interpret clusters using original, unscaled physical measurements.
* Compare clustering techniques across algorithmic, operational, and deployment criteria.
* Save the trained K-Means model bundle using `joblib` for deployment.
* Deploy the model using a modern, interactive Streamlit web dashboard.
* Discuss ethical considerations and technical limitations of clustering in healthcare.

# 4. Import Libraries & Global Configuration

We import the essential Python libraries for data processing (`numpy`, `pandas`), visualization (`matplotlib`, `seaborn`), unsupervised machine learning (`scikit-learn`), hierarchical clustering (`scipy`), and model serialization (`joblib`).

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import linkage, dendrogram

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Set reproducible random state
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Libraries imported successfully. RANDOM_STATE fixed at {RANDOM_STATE}.")

: 

# 5. Synthetic Data Generation

Because no single public dataset contains all required clinical, lifestyle, and claim indicators together, we generate a synthetic dataset of **1,200 policyholders** using `numpy.random.default_rng` with a fixed seed (`RANDOM_STATE = 42`).

Three broad latent physiological profiles are modeled:
1. **Profile 1 – Lower indicators:** Younger adults, normal BMI, optimal blood pressure, low total cholesterol, high physical activity, minimal historical claims.
2. **Profile 2 – Moderate indicators:** Middle-aged adults, overweight BMI range, pre-hypertensive blood pressure, borderline cholesterol, moderate activity, moderate claims.
3. **Profile 3 – Higher indicators:** Older adults, elevated BMI (obesity range), hypertensive blood pressure, high cholesterol, sedentary lifestyle, more frequent claims.

**Crucial Note:** No `risk_group` or label column is stored in the dataset. The machine learning algorithms must discover these groupings purely through unsupervised learning.

In [ ]:
N_SAMPLES = 1200
rng = np.random.default_rng(RANDOM_STATE)

# Allocate sample counts across the three latent profiles
nA = 420
nB = 420
nC = N_SAMPLES - (nA + nB)

records = []

# Profile 1: Lower-indicator profile
records.append({
    "age": rng.normal(30, 6, nA).clip(18, 48),
    "bmi": rng.normal(21.5, 2.2, nA).clip(16, 26),
    "systolic_bp": rng.normal(110, 6, nA).clip(92, 126),
    "diastolic_bp": rng.normal(70, 5, nA).clip(56, 82),
    "cholesterol": rng.normal(165, 16, nA).clip(115, 205),
    "physical_activity": rng.normal(8.5, 1.8, nA).clip(5.0, 12.0),
    "previous_claims": rng.poisson(0.8, nA).clip(0, 3),
})

# Profile 2: Moderate-indicator profile
records.append({
    "age": rng.normal(49, 7, nB).clip(34, 64),
    "bmi": rng.normal(27.8, 2.4, nB).clip(22, 34),
    "systolic_bp": rng.normal(132, 7, nB).clip(116, 148),
    "diastolic_bp": rng.normal(84, 5, nB).clip(72, 94),
    "cholesterol": rng.normal(218, 18, nB).clip(175, 260),
    "physical_activity": rng.normal(4.2, 1.2, nB).clip(1.8, 7.0),
    "previous_claims": rng.poisson(2.6, nB).clip(0, 6),
})

# Profile 3: Higher-indicator profile
records.append({
    "age": rng.normal(68, 7, nC).clip(52, 85),
    "bmi": rng.normal(34.5, 3.2, nC).clip(28, 45),
    "systolic_bp": rng.normal(156, 9, nC).clip(136, 190),
    "diastolic_bp": rng.normal(98, 6, nC).clip(84, 120),
    "cholesterol": rng.normal(275, 20, nC).clip(230, 330),
    "physical_activity": rng.normal(1.2, 0.8, nC).clip(0.0, 3.5),
    "previous_claims": rng.poisson(6.2, nC).clip(2, 12),
})

# Combine and shuffle so records appear in completely random order
df = pd.concat([pd.DataFrame(r) for r in records], ignore_index=True)
df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

# Format features to clinically meaningful precisions
df["age"] = df["age"].round(0).astype(int)
df["bmi"] = df["bmi"].round(1)
df["systolic_bp"] = df["systolic_bp"].round(0).astype(int)
df["diastolic_bp"] = df["diastolic_bp"].round(0).astype(int)
df["cholesterol"] = df["cholesterol"].round(0).astype(int)
df["physical_activity"] = df["physical_activity"].round(1)
df["previous_claims"] = df["previous_claims"].astype(int)

# Insert unique Policyholder Identifier
df.insert(0, "policyholder_id", [f"PH{str(i+1).zfill(4)}" for i in range(len(df))])

print(f"Dataset successfully created with {len(df)} records.")

# 6. Initial DataFrame Inspection

We inspect the top rows, dataset dimensions, and column data types using `.head()`, `.shape`, and `.info()`.

In [ ]:
# Display the first 5 records
df.head()

In [ ]:
# Display dataset shape
print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Display column data types and memory usage
df.info()

# 7. Dataset Description

The dataset comprises 1 identifier and 7 numerical health attributes:

| Feature Name | Type | Physical Units | Description | Expected Range |
|:---|:---|:---|:---|:---|
| `policyholder_id` | Categorical / String | N/A | Unique identifier assigned to each insured individual | `PH0001` – `PH1200` |
| `age` | Integer | Years | Chronological age of the policyholder | 18 – 85 years |
| `bmi` | Float | $\text{kg/m}^2$ | Body Mass Index (weight in kg / height in $\text{m}^2$) | 15.0 – 45.0 |
| `systolic_bp` | Integer | mmHg | Maximum arterial pressure during heart contraction | 90 – 190 mmHg |
| `diastolic_bp` | Integer | mmHg | Minimum arterial pressure between heartbeats | 55 – 120 mmHg |
| `cholesterol` | Integer | mg/dL | Serum total cholesterol level | 110 – 330 mg/dL |
| `physical_activity`| Float | Hours / Week | Moderate-to-vigorous physical exercise per week | 0.0 – 12.0 hours |
| `previous_claims` | Integer | Count | Number of prior insurance medical claims submitted | 0 – 12 claims |

Below are the descriptive summary statistics computed on all numerical features.

In [ ]:
# Generate descriptive summary statistics
df.describe().round(2)

# 8. Introduce Missing Values

In real-world insurance and healthcare records, clinical measurements are frequently omitted due to non-attendance, laboratory delays, or non-reporting. To simulate this realistic challenge, we introduce approximately **3% missing values (`NaN`)** into the four clinical measurement columns:
* `bmi`
* `systolic_bp`
* `diastolic_bp`
* `cholesterol`

### Why Missing Values Must Be Handled Before Clustering
Distance-based clustering algorithms such as K-Means and Hierarchical Clustering compute pairwise Euclidean distances between vectors:
$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{j=1}^{d} (x_j - y_j)^2}$$
If any component $x_j$ is `NaN`, the distance calculation evaluates to `NaN`, preventing centroid assignment and distance matrix computation. Therefore, systematic handling of missing values is an indispensable preprocessing requirement.

In [ ]:
MISSING_RATE = 0.03
COLS_WITH_MISSING = ["bmi", "systolic_bp", "diastolic_bp", "cholesterol"]

df_missing = df.copy()

for col in COLS_WITH_MISSING:
    n_missing = int(len(df_missing) * MISSING_RATE)
    missing_indices = rng.choice(df_missing.index, size=n_missing, replace=False)
    df_missing.loc[missing_indices, col] = np.nan

# Display missing values count
missing_summary = pd.DataFrame({
    "Missing Count": df_missing.isnull().sum(),
    "Missing Percentage (%)": (df_missing.isnull().sum() / len(df_missing) * 100).round(2)
})
missing_summary

In [ ]:
# Visualize missing value counts across features
plt.figure(figsize=(8, 4))
missing_counts = df_missing.isnull().sum()
missing_counts[missing_counts > 0].plot(kind="bar", color="#E65100", edgecolor="black", alpha=0.85)
plt.title("Missing Value Count Across Clinical Indicators (~3% Injected)", fontsize=13, fontweight="bold")
plt.xlabel("Feature Name", fontsize=11)
plt.ylabel("Number of Missing Records", fontsize=11)
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# 9. Data Cleaning & Median Imputation

Instead of discarding rows with missing values (which causes loss of statistical power and potential selection bias), we impute missing clinical values using `SimpleImputer(strategy="median")`.

### Rationale for Median Imputation:
Clinical parameters such as blood pressure and cholesterol can have skewed distributions with physiological extremes. The **median** is a robust measure of central tendency that is resistant to outlier distortion, making it superior to the arithmetic mean for clinical parameter imputation.

The imputer is fitted on the dataset and saved for identical deployment during real-time inference.

In [ ]:
features = [
    "age",
    "bmi",
    "systolic_bp",
    "diastolic_bp",
    "cholesterol",
    "physical_activity",
    "previous_claims"
]

# Fit SimpleImputer on the features
imputer = SimpleImputer(strategy="median")
df_imputed = df_missing.copy()
df_imputed[features] = imputer.fit_transform(df_missing[features])

# Compare missing values before and after imputation
comparison_df = pd.DataFrame({
    "Before Imputation (NaN)": df_missing[features].isnull().sum(),
    "After Imputation (NaN)": df_imputed[features].isnull().sum()
})
print("Missing Value Verification:")
display(comparison_df)

# Assign cleaned dataset to df
df = df_imputed.copy()

# 10. Exploratory Data Analysis (EDA)

We explore the distributions of all numerical features through dedicated histograms with Kernel Density Estimates (KDE) to assess central tendencies, spreads, and multi-modal tendencies.

In [ ]:
# 1. Age distribution
plt.figure(figsize=(8, 4))
sns.histplot(df["age"], bins=30, kde=True, color="#1565C0", edgecolor="white")
plt.title("Age Distribution of Policyholders", fontsize=13, fontweight="bold")
plt.xlabel("Age (years)", fontsize=11)
plt.ylabel("Policyholder Count", fontsize=11)
plt.show()

In [ ]:
# 2. BMI distribution
plt.figure(figsize=(8, 4))
sns.histplot(df["bmi"], bins=30, kde=True, color="#2E7D32", edgecolor="white")
plt.title("Body Mass Index (BMI) Distribution", fontsize=13, fontweight="bold")
plt.xlabel("BMI (kg/m²)", fontsize=11)
plt.ylabel("Policyholder Count", fontsize=11)
plt.show()

In [ ]:
# 3. Blood pressure distributions (Systolic & Diastolic)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["systolic_bp"], bins=30, kde=True, color="#C62828", edgecolor="white", ax=axes[0])
axes[0].set_title("Systolic Blood Pressure Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Systolic BP (mmHg)", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)

sns.histplot(df["diastolic_bp"], bins=30, kde=True, color="#AD1457", edgecolor="white", ax=axes[1])
axes[1].set_title("Diastolic Blood Pressure Distribution", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Diastolic BP (mmHg)", fontsize=11)
axes[1].set_ylabel("Count", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Cholesterol distribution
plt.figure(figsize=(8, 4))
sns.histplot(df["cholesterol"], bins=30, kde=True, color="#F57C00", edgecolor="white")
plt.title("Total Serum Cholesterol Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Cholesterol (mg/dL)", fontsize=11)
plt.ylabel("Policyholder Count", fontsize=11)
plt.show()

In [ ]:
# 5. Physical activity distribution
plt.figure(figsize=(8, 4))
sns.histplot(df["physical_activity"], bins=25, kde=True, color="#00838F", edgecolor="white")
plt.title("Weekly Physical Activity Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Physical Activity (hours/week)", fontsize=11)
plt.ylabel("Policyholder Count", fontsize=11)
plt.show()

In [ ]:
# 6. Previous claims distribution
plt.figure(figsize=(8, 4))
sns.histplot(df["previous_claims"], discrete=True, color="#4527A0", edgecolor="white")
plt.title("Previous Claims Frequency Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Previous Claims Count", fontsize=11)
plt.ylabel("Policyholder Count", fontsize=11)
plt.show()

# 11. Boxplots for Numerical Features

Boxplots illustrate the five-number summary (minimum, first quartile $Q_1$, median, third quartile $Q_3$, and maximum) along with potential outliers beyond $1.5 \times \text{IQR}$. They also reveal the drastic differences in scale across features (e.g., cholesterol in hundreds vs claims in single digits).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(y=df[col], ax=axes[i], color="#42A5F5", width=0.4,
                boxprops=dict(alpha=0.85),
                medianprops=dict(color="red", linewidth=2))
    axes[i].set_title(col.replace("_", " ").title(), fontsize=11, fontweight="bold")
    axes[i].set_ylabel("Value")
    axes[i].grid(axis="y", linestyle="--", alpha=0.5)

# Hide unused 8th subplot
axes[7].set_visible(False)
plt.suptitle("Feature Boxplots (Distribution Spread & Scale Differences)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 12. Correlation Heatmap

We compute Pearson's pairwise correlation matrix across all numerical indicators to observe inter-variable linear dependencies.

> **Important Note:** Correlation indicates statistical association, **NOT** biological or epidemiological causation.

In [ ]:
corr_matrix = df[features].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.8,
    cbar_kws={"shrink": 0.8}
)
plt.title("Pearson Correlation Heatmap Between Health Indicators", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

# 13. Feature Selection

We extract the feature matrix $\mathbf{X}$ containing only the 7 clinical and lifestyle indicators required for unsupervised clustering, discarding the non-informative `policyholder_id`.

In [ ]:
features = [
    "age",
    "bmi",
    "systolic_bp",
    "diastolic_bp",
    "cholesterol",
    "physical_activity",
    "previous_claims"
]

X = df[features].copy()
print(f"Feature matrix X prepared with shape: {X.shape}")
display(X.head())

# 14. Feature Scaling (StandardScaler)

### Why Feature Scaling is Essential for Clustering:
Both K-Means and Hierarchical Clustering rely on geometric Euclidean distance:
$$d(\mathbf{u}, \mathbf{v}) = \sqrt{ \sum_{j=1}^{d} (u_j - v_j)^2 }$$
Our features have completely different physical dimensions:
* `age`: 18 – 85 years (range $\approx 67$)
* `systolic_bp`: 90 – 190 mmHg (range $\approx 100$)
* `cholesterol`: 110 – 330 mg/dL (range $\approx 220$)
* `physical_activity`: 0 – 12 hours (range $\approx 12$)
* `previous_claims`: 0 – 12 count (range $\approx 12$)

Without scaling, high-magnitude features like `cholesterol` would dominate the squared Euclidean distance, rendering lower-magnitude features like `previous_claims` virtually invisible to the algorithm.

`StandardScaler` standardizes each feature to have **zero mean and unit variance**:
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Compare before and after scaling for the first 3 records
print("Features Before Scaling (Original Units):")
display(X.head(3))

print("\nFeatures After StandardScaler (Mean=0, Std=1):")
display(pd.DataFrame(X_scaled, columns=features).head(3).round(3))

# 15. K-Means Clustering Across $K = 2 \dots 8$

K-Means partitions the observations into $K$ clusters such that the Within-Cluster Sum of Squares (WCSS / Inertia) is minimized:
$$\text{WCSS} = \sum_{k=1}^{K} \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$$

We fit K-Means models for $K \in [2, 8]$ using `n_init=20` to ensure convergence to optimal global minima, calculating WCSS and the Silhouette Score for each $K$.

In [ ]:
k_range = list(range(2, 9))
wcss_scores = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_scaled)
    wcss = km.inertia_
    sil = silhouette_score(X_scaled, labels)
    wcss_scores.append(wcss)
    silhouette_scores.append(sil)

metrics_df = pd.DataFrame({
    "Number of Clusters (K)": k_range,
    "WCSS (Inertia)": np.round(wcss_scores, 2),
    "Silhouette Score": np.round(silhouette_scores, 4)
})

display(metrics_df)

# 16. The Elbow Method (WCSS vs $K$)

The Elbow Method identifies the value of $K$ where the reduction in WCSS exhibits diminishing marginal returns (forming an "elbow" or inflection point).

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(k_range, wcss_scores, "o-", color="#1565C0", linewidth=2.5, markersize=8, label="WCSS")
plt.axvline(3, color="#D32F2F", linestyle="--", linewidth=1.8, label="Elbow at K=3")
plt.scatter([3], [wcss_scores[1]], color="#D32F2F", s=150, zorder=5)

plt.title("Elbow Method: Number of Clusters vs WCSS", fontsize=13, fontweight="bold")
plt.xlabel("Number of Clusters (K)", fontsize=11)
plt.ylabel("WCSS (Within-Cluster Sum of Squares)", fontsize=11)
plt.xticks(k_range)
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# 17. Silhouette Score Analysis

The Silhouette Score measures how similar a sample is to its assigned cluster (cohesion) compared to neighboring clusters (separation):
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
Where:
* $a(i)$ = mean intra-cluster distance of point $i$.
* $b(i)$ = mean nearest-cluster distance of point $i$.

The score ranges from **$-1$ to $+1$**:
* Values near $+1$ indicate excellent cluster cohesion and distinct cluster separation.
* Values near $0$ indicate overlapping clusters.
* Negative values indicate points assigned to the wrong cluster.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(k_range, silhouette_scores, "s-", color="#2E7D32", linewidth=2.5, markersize=8, label="Silhouette Score")
plt.axvline(3, color="#D32F2F", linestyle="--", linewidth=1.8, label="Optimal at K=3")
plt.scatter([3], [silhouette_scores[1]], color="#D32F2F", s=150, zorder=5)

plt.title("Silhouette Analysis: Number of Clusters vs Silhouette Score", fontsize=13, fontweight="bold")
plt.xlabel("Number of Clusters (K)", fontsize=11)
plt.ylabel("Silhouette Score", fontsize=11)
plt.xticks(k_range)
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# 18. Selection of Optimal Number of Clusters ($K$)

Based on the quantitative metrics and domain interpretability:
1. **WCSS Reduction:** A substantial decrease in WCSS occurs from $K=2$ ($3425.32$) to $K=3$ ($1707.00$), representing a dramatic drop of $1718.32$. Moving beyond $K=3$ yields modest decreases ($K=4$ WCSS is $1524.66$, a drop of only $182.34$).
2. **Silhouette Maximization:** $K=3$ achieves the peak Silhouette Score ($0.4987$), outperforming all other tested cluster counts.
3. **Clinical / Business Interpretability:** Three clusters map directly onto three actionable preventive risk tiers: **Lower-indicator**, **Moderate-indicator**, and **Higher-indicator** profiles.

Therefore, **$K = 3$** is formally justified and selected.

In [ ]:
optimal_k = 3
print(f"Optimal cluster count confirmed: optimal_k = {optimal_k}")

# 19. Final K-Means Model Training

We fit the final K-Means model with $K = 3$ and append the resulting cluster labels to the dataset.

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=20)
kmeans_labels = kmeans.fit_predict(X_scaled)

df_with_clusters = df.copy()
df_with_clusters["cluster"] = kmeans_labels

# Display cluster policyholder counts
cluster_counts = df_with_clusters["cluster"].value_counts().sort_index()
print("Cluster Sizes (Policyholder Counts):")
display(cluster_counts.to_frame(name="Policyholders"))

# 20. Cluster Size Visualization

We visualize the distribution of policyholders across the discovered clusters using a bar chart.

In [ ]:
plt.figure(figsize=(7, 4.5))
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = plt.bar([f"Cluster {i}" for i in cluster_counts.index], cluster_counts.values,
               color=colors, edgecolor="black", width=0.5, alpha=0.85)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{int(yval)} ({yval/len(df)*100:.1f}%)",
             ha="center", va="bottom", fontweight="bold")

plt.title("Policyholder Distribution Across Discovered Clusters (K=3)", fontsize=13, fontweight="bold")
plt.xlabel("Cluster", fontsize=11)
plt.ylabel("Number of Policyholders", fontsize=11)
plt.ylim(0, max(cluster_counts.values) + 60)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# 21. Agglomerative Hierarchical Clustering

As a comparative baseline, we implement Agglomerative Hierarchical Clustering with **Ward linkage**, which minimizes the total within-cluster variance at each successive bottom-up merge.

In [ ]:
agg_clustering = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward")
hc_labels = agg_clustering.fit_predict(X_scaled)

hc_sil = silhouette_score(X_scaled, hc_labels)
hc_counts = pd.Series(hc_labels).value_counts().sort_index()

print(f"Hierarchical Clustering Silhouette Score (K={optimal_k}): {hc_sil:.4f}")
print("Hierarchical Cluster Sizes:")
display(hc_counts.to_frame(name="Policyholders"))

# 22. Hierarchical Clustering Dendrogram

The dendrogram depicts the tree of hierarchical merges computed using `scipy.cluster.hierarchy.linkage`. The horizontal cut height demonstrates how cutting the tree at a linkage distance threshold naturally yields 3 primary branches.

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(13, 6))
dend = dendrogram(
    Z,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=45,
    leaf_font_size=10,
    color_threshold=0,
    above_threshold_color="#1565C0"
)

# Compute horizontal threshold height for 3 clusters
distances = sorted(Z[:, 2], reverse=True)
cut_threshold = (distances[optimal_k - 2] + distances[optimal_k - 1]) / 2.0
plt.axhline(y=cut_threshold, color="#D32F2F", linestyle="--", linewidth=1.8,
            label=f"Cut Height = {cut_threshold:.1f} (Yields K=3)")

plt.title("Dendrogram – Agglomerative Hierarchical Clustering (Ward Linkage, Last 30 Merges)",
          fontsize=13, fontweight="bold")
plt.xlabel("Cluster Index / Truncated Merges", fontsize=11)
plt.ylabel("Ward Linkage Distance", fontsize=11)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# 23. Comparison: K-Means vs Hierarchical Clustering

We compare the performance metrics, cluster size distributions, and deployment properties of both algorithms.

In [ ]:
comparison_metrics = pd.DataFrame([
    {
        "Method": "K-Means",
        "Number of Clusters": optimal_k,
        "Silhouette Score": round(metrics_df.loc[metrics_df["Number of Clusters (K)"] == optimal_k, "Silhouette Score"].values[0], 4),
        "Cluster 0 Size": int(cluster_counts.get(0, 0)),
        "Cluster 1 Size": int(cluster_counts.get(1, 0)),
        "Cluster 2 Size": int(cluster_counts.get(2, 0)),
        "Direct Inference (.predict)": "Yes (Nearest Centroid)",
        "Deployment Suitability": "High (Production Model)"
    },
    {
        "Method": "Hierarchical (Ward)",
        "Number of Clusters": optimal_k,
        "Silhouette Score": round(hc_sil, 4),
        "Cluster 0 Size": int(hc_counts.get(0, 0)),
        "Cluster 1 Size": int(hc_counts.get(1, 0)),
        "Cluster 2 Size": int(hc_counts.get(2, 0)),
        "Direct Inference (.predict)": "No (Requires Full Refit)",
        "Deployment Suitability": "Low (Comparison Baseline Only)"
    }
])

display(comparison_metrics)

# 24. Dimensionality Reduction (PCA)

We apply Principal Component Analysis (PCA) to project the 7-dimensional standardized feature space into 2 orthogonal components for visualization.

> **Clarification:** PCA is applied **strictly for dimensionality reduction and 2D visualization**, NOT as the clustering algorithm itself.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

ev = pca.explained_variance_ratio_
total_ev = np.sum(ev)

pca_variance_df = pd.DataFrame({
    "Component": ["Principal Component 1 (PC1)", "Principal Component 2 (PC2)", "Total (PC1 + PC2)"],
    "Explained Variance Ratio": [f"{ev[0]*100:.2f}%", f"{ev[1]*100:.2f}%", f"{total_ev*100:.2f}%"]
})
display(pca_variance_df)

# 25. PCA 2D Cluster Visualization

We plot policyholders in the 2D subspace defined by PC1 and PC2, coloring each point according to its assigned K-Means cluster.

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=kmeans_labels,
    cmap="tab10",
    s=40,
    alpha=0.75,
    edgecolors="white",
    linewidths=0.3
)

# Plot centroids projected into PCA space
for k in range(optimal_k):
    cx = X_pca[kmeans_labels == k, 0].mean()
    cy = X_pca[kmeans_labels == k, 1].mean()
    plt.scatter(cx, cy, marker="X", s=220, color="black", zorder=10)
    plt.annotate(f"Centroid {k}", (cx, cy), fontsize=10, fontweight="bold",
                 xytext=(8, 6), textcoords="offset points",
                 bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8))

plt.title(f"PCA 2D Cluster Projection (Total Variance Explained: {total_ev*100:.1f}%)",
          fontsize=13, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({ev[0]*100:.1f}% variance)", fontsize=11)
plt.ylabel(f"Principal Component 2 ({ev[1]*100:.1f}% variance)", fontsize=11)
plt.legend(handles=scatter.legend_elements()[0], labels=[f"Cluster {i}" for i in range(optimal_k)],
           title="K-Means Cluster", loc="best")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# 26. Cluster Profiling Using Original Feature Values

To make the discovered clusters human-interpretable for healthcare professionals, we compute the mean characteristics of each cluster using the **ORIGINAL (unscaled) feature values**.

In [ ]:
# Compute cluster means on original physical units
cluster_profile = df_with_clusters.groupby("cluster")[features].mean().round(2)
cluster_profile["size"] = cluster_counts.values

# Reorder columns with size prominent
cols_order = ["size"] + features
cluster_profile = cluster_profile[cols_order]

print("Cluster Profiles (Centroids in Original Physical Units):")
display(cluster_profile)

# 27. Cluster Interpretation & Dynamic Profile Naming

Because K-Means assigns cluster integer IDs arbitrarily, we compute a composite health indicator score to dynamically assign descriptive profile names:
$$\text{Composite Score} = \overline{\text{age}} + \overline{\text{bmi}} + \overline{\text{systolic\_bp}} + \overline{\text{cholesterol}}$$
Clusters are then mapped to descriptive names:
* **Lower-indicator profile** (lowest composite indicators)
* **Moderate-indicator profile** (intermediate indicators)
* **Higher-indicator profile** (highest composite indicators)

In [ ]:
# Compute composite score for ranking
composite_scores = (
    cluster_profile["age"] +
    cluster_profile["bmi"] +
    cluster_profile["systolic_bp"] +
    cluster_profile["cholesterol"]
)

# Rank clusters
sorted_clusters = composite_scores.sort_values().index.tolist()
label_names = ["Lower-indicator profile", "Moderate-indicator profile", "Higher-indicator profile"]

cluster_names = {cluster_id: label_names[rank] for rank, cluster_id in enumerate(sorted_clusters)}

cluster_profile_named = cluster_profile.copy()
cluster_profile_named["profile_name"] = cluster_profile_named.index.map(cluster_names)

print("Dynamic Cluster Mapping:")
for cid, name in cluster_names.items():
    print(f"  Cluster {cid} -> {name}")

display(cluster_profile_named)

### Detailed Clinical Profile Interpretations:

* **Lower-indicator profile:**
  * **Characteristics:** Youngest cohort (average age ~30 years), healthy BMI (~21.3), optimal blood pressure (~110/70 mmHg), low cholesterol (~163 mg/dL), highest weekly exercise (~8.6 hrs/week), and negligible claims (average 0.8 claims).
  * **Preventive Focus:** Maintaining regular physical activity, balanced nutrition, routine annual preventive health screenings, and healthy lifestyle preservation.

* **Moderate-indicator profile:**
  * **Characteristics:** Middle-aged cohort (average age ~49 years), overweight BMI range (~27.7), pre-hypertensive blood pressure (~132/84 mmHg), borderline cholesterol (~218 mg/dL), moderate activity (~4.2 hrs/week), and moderate claims (~2.5 claims).
  * **Preventive Focus:** Increasing physical activity, balanced dietary adjustments, healthy weight management, and routine health screenings.

* **Higher-indicator profile:**
  * **Characteristics:** Older cohort (average age ~68 years), elevated BMI in obesity range (~34.2), hypertensive blood pressure (~157/98 mmHg), high cholesterol (~275 mg/dL), sedentary lifestyle (~1.2 hrs/week), and frequent insurance claims (~6.2 claims).
  * **Preventive Focus:** Regular clinical reviews with medical professionals, appropriate physician-guided physical activity, nutrition and weight management, and routine cardiovascular and metabolic monitoring.

# 28. Cluster Profile Visualizations

We visualize the differences across clusters using multi-panel bar charts and a standardized profile heatmap.

In [ ]:
# Multi-panel feature comparison across clusters
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

cluster_labels = [f"C{i}: {cluster_names[i].split()[0]}" for i in cluster_profile.index]

for i, col in enumerate(features):
    vals = cluster_profile[col].values
    bars = axes[i].bar(cluster_labels, vals, color=colors, edgecolor="black", alpha=0.85)
    axes[i].set_title(col.replace("_", " ").title(), fontsize=11, fontweight="bold")
    axes[i].grid(axis="y", linestyle="--", alpha=0.5)
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2.0, h*0.5, f"{h:.1f}",
                     ha="center", va="center", color="white", fontweight="bold")

axes[7].set_visible(False)
plt.suptitle("Cluster-wise Feature Centroids (Original Physical Units)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Standardized feature heatmap across clusters (Z-score normalized)
scaled_profile = pd.DataFrame(
    StandardScaler().fit_transform(cluster_profile[features]),
    index=[f"Cluster {i} ({cluster_names[i]})" for i in cluster_profile.index],
    columns=[f.replace("_", " ").title() for f in features]
)

plt.figure(figsize=(10, 4))
sns.heatmap(scaled_profile, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=1.0)
plt.title("Standardized Relative Health Indicators Across Clusters (Z-Score Heatmap)", fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

# 29. Final Model Metrics Summary

Below is the definitive summary of performance metrics for the selected models and dimensionality reduction components.

In [ ]:
final_metrics_summary = pd.DataFrame({
    "Evaluation Metric": [
        "Selected Number of Clusters (K)",
        "K-Means WCSS (Inertia)",
        "K-Means Silhouette Score",
        "Hierarchical Clustering Silhouette Score",
        "PCA Component 1 Explained Variance",
        "PCA Component 2 Explained Variance",
        "Total PCA Explained Variance (2 Components)"
    ],
    "Value": [
        str(optimal_k),
        f"{metrics_df.loc[metrics_df['Number of Clusters (K)'] == optimal_k, 'WCSS (Inertia)'].values[0]:.2f}",
        f"{metrics_df.loc[metrics_df['Number of Clusters (K)'] == optimal_k, 'Silhouette Score'].values[0]:.4f}",
        f"{hc_sil:.4f}",
        f"{ev[0]*100:.2f}%",
        f"{ev[1]*100:.2f}%",
        f"{total_ev*100:.2f}%"
    ]
})

display(final_metrics_summary)

# 30. Save Model Bundle for Deployment

We serialize the complete deployment artifact package into `models/model_bundle.joblib` using `joblib`. The deployment application loads this single bundle to process and segment new incoming policyholder records without retraining.

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)

# Prepare cluster profiles DataFrame
cluster_profiles_df = cluster_profile.copy().reset_index()

model_bundle = {
    "model": kmeans,
    "imputer": imputer,
    "scaler": scaler,
    "pca": pca,
    "features": features,
    "selected_k": optimal_k,
    "cluster_profile": cluster_profile,
    "cluster_names": cluster_names,
    # Supplementary aliases for full app compatibility
    "cluster_profiles": cluster_profiles_df,
    "profile_name_map": cluster_names,
    "seed": RANDOM_STATE
}

joblib.dump(model_bundle, "models/model_bundle.joblib")
print("Model bundle saved successfully to 'models/model_bundle.joblib'.")

# Also update report CSVs
cluster_profiles_df.to_csv("reports/cluster_profiles.csv", index=False)
cluster_profile_named.reset_index().to_csv("reports/cluster_profiles_named.csv", index=False)
cluster_counts.to_frame(name="size").reset_index().rename(columns={"index": "cluster"}).to_csv("reports/cluster_sizes.csv", index=False)
metrics_df.to_csv("reports/model_metrics.csv", index=False)
print("Reports CSVs updated in 'reports/'.")

# 31. Test New Policyholder Inference Pipeline

We demonstrate end-to-end inference on a hypothetical new policyholder record:
* **Age:** 52 years
* **BMI:** 31.0 $\text{kg/m}^2$
* **Systolic BP:** 145 mmHg
* **Diastolic BP:** 90 mmHg
* **Cholesterol:** 230 mg/dL
* **Physical Activity:** 2.5 hours/week
* **Previous Claims:** 4 claims

The inference workflow applies the saved `SimpleImputer`, transforms with `StandardScaler`, queries `kmeans.predict()`, maps the predicted cluster ID to its descriptive profile name, and contrasts the policyholder's values against the cluster centroid.

In [ ]:
# Load the saved model bundle to verify standalone portability
loaded_bundle = joblib.load("models/model_bundle.joblib")

test_policyholder = {
    "age": 52,
    "bmi": 31.0,
    "systolic_bp": 145,
    "diastolic_bp": 90,
    "cholesterol": 230,
    "physical_activity": 2.5,
    "previous_claims": 4
}

test_df = pd.DataFrame([test_policyholder])[loaded_bundle["features"]]

# Apply preprocessing pipeline
X_test_imp = loaded_bundle["imputer"].transform(test_df)
X_test_scl = loaded_bundle["scaler"].transform(X_test_imp)

# Predict cluster
predicted_cluster = int(loaded_bundle["model"].predict(X_test_scl)[0])
assigned_profile = loaded_bundle["cluster_names"][predicted_cluster]

# Retrieve cluster average values
cluster_centroid = loaded_bundle["cluster_profile"].loc[predicted_cluster]

print("=" * 60)
print("NEW POLICYHOLDER INFERENCE RESULT")
print("=" * 60)
print(f"Assigned Cluster : Cluster {predicted_cluster}")
print(f"Profile Name     : {assigned_profile}")
print("-" * 60)

# Build explainability comparison table
inference_comparison = pd.DataFrame({
    "Indicator": [f.replace("_", " ").title() for f in loaded_bundle["features"]],
    "Policyholder Value": [test_policyholder[f] for f in loaded_bundle["features"]],
    "Cluster Average": [cluster_centroid[f] for f in loaded_bundle["features"]],
    "Difference": [round(test_policyholder[f] - cluster_centroid[f], 1) for f in loaded_bundle["features"]]
})

display(inference_comparison)

# 32. Ethical Considerations

Applying unsupervised machine learning to health and insurance contexts raises significant ethical, societal, and regulatory considerations:

### 1. Synthetic Data Notice
The dataset analyzed herein is completely synthetic and generated algorithmically. It does not represent any real patient population, and its findings must not be extrapolated to clinical diagnostics or actuarial pricing.

### 2. Medical Interpretation & Non-Diagnostic Scope
Unsupervised clusters are mathematical groupings reflecting multidimensional variance. They are **not medical diagnoses**, pathology assessments, or clinical prognoses.

### 3. Insurance Fairness & Non-Discrimination
Health risk segmentation should be utilized exclusively for **positive preventive engagement** (e.g., proactive coaching, wellness discounts, screening invitations). These algorithms must **never** be used punitively to deny insurance coverage, increase premiums, restrict healthcare benefits, or cancel policies.

### 4. Demographic & Socioeconomic Bias
Real-world clinical data often mirrors structural disparities in healthcare access, socioeconomic status, and historical reporting biases. Models trained on biased data risk entrenching health disparities.

### 5. Patient Privacy & Security
In real-world implementations, clinical data must adhere to strict regulatory compliance standards including **HIPAA** (Health Insurance Portability and Accountability Act), **GDPR** (General Data Protection Regulation), and local privacy laws, requiring robust data anonymization and encryption.

### 6. Stigmatization
Labels such as "higher-indicator" or "at-risk" can impose psychosocial stigma on individuals. Healthcare organizations must communicate findings using constructive, wellness-centered language.

### 7. Inaccuracies in Self-Reported Data
Lifestyle measures such as weekly physical activity and BMI are frequently self-reported in wellness programs, introducing potential self-reporting and recall bias.

# 33. Project & Methodological Limitations

1. **Synthetic Nature:** The dataset is artificially synthesized and may not mirror the nuanced correlation structures of real clinical populations.
2. **Feature Set Size:** The analysis is restricted to 7 indicators; other critical health determinants (e.g., HbA1c, family history, diet, smoking status, genomics) are absent.
3. **Absence of Real Clinical Validation:** Cluster validity has not been benchmarked against longitudinal clinical outcomes.
4. **Distance Metric Assumptions:** K-Means assumes spherical clusters of approximately equal variance and size, which may not always hold true.
5. **Heuristic Nature of K Selection:** While guided by the Elbow Method and Silhouette Scores, selecting $K$ remains partially subjective.
6. **Information Loss in PCA:** While 2D PCA captures ~87.5% of variance, ~12.5% of multidimensional geometric structure is discarded in visualization.
7. **Non-Causality:** Clustering reveals correlation and numerical proximity; it does not prove causal mechanisms between indicators.
8. **Missing Data Assumptions:** Median imputation assumes missing values occur at random (MCAR/MAR), which may not hold in clinical practice.
9. **Sensitivity to Initial Conditions:** K-Means can converge to local minima, mitigated here by setting `n_init=20`.
10. **Generalizability:** Models trained on single insurance cohorts do not automatically generalize to diverse demographic populations.

# 34. Final Conclusion

* **Optimal Cohorts Discovered:** Unsupervised analysis identified $K = 3$ well-separated, clinically interpretable clusters supported by a peak Silhouette Score of **0.4987** and a prominent WCSS elbow.
* **Distinct Behavioral Profiles:** The three cohorts naturally correspond to:
  * **Lower-indicator Profile:** Young, active, normal physiological markers.
  * **Moderate-indicator Profile:** Middle-aged with borderline indicators.
  * **Higher-indicator Profile:** Older cohort with elevated cardiovascular and metabolic indicators.
* **Enabling Preventive Healthcare:** This segmentation enables insurance providers to deploy targeted wellness programmes, transitioning from passive claims reimbursement to proactive health preservation.
* **Deployment Validation:** The model pipeline was serialized, tested on new policyholder data, and connected to an interactive Streamlit application.
* **Prerequisite for Clinical Use:** Any clinical translation requires rigorous real-world validation, domain clinician oversight, and strict adherence to healthcare privacy and fairness principles.

# 35. Streamlit Deployment Overview

The trained model bundle (`models/model_bundle.joblib`) powers the interactive web dashboard located in `app/app.py`.

### How to Launch the Web Application:
```bash
streamlit run app/app.py
```

### Dashboard Capabilities:
* **Interactive Indicator Form:** Allows input of age, BMI, blood pressure, cholesterol, physical activity, and claims.
* **Instant Group Assignment:** Classifies policyholders via the fitted K-Means pipeline.
* **Explainable Comparison:** Displays a side-by-side contrast between the policyholder's indicators and the cluster average.
* **Tailored Preventive Programme:** Displays tailored, generic educational preventive guidelines with medical disclaimers.
* **Model Analysis Visualizations:** Interactive tabs displaying the Elbow curve, Silhouette curve, PCA cluster scatter plot, and Dendrogram.